# 03 – Plans and Allowances

ducto supports subscription-style plans with free monthly allowances.
The v2 pricing config embeds plan definitions alongside model formulas.

> **Note:** Uses `MemoryStore` because plan management via PostgresStore
> requires pre-seeded `credit_plans` table rows. MemoryStore handles
> plan definitions inline.

In [1]:
import uuid
from datetime import datetime, timedelta
from ducto.interface.memory import MemoryStore
from ducto.manager import CreditManager
from ducto.engine import PricingEngine
from ducto.metrics import UsageMetrics, ToolCall
from ducto.interface.models import (
    PricingConfigData, PlanDefinition,
    CreditMetadata, SpendCap,
)

store = MemoryStore()
store.setup()
print("✔ MemoryStore ready.")

✔ MemoryStore ready.


### Persist plan definitions in pricing config v2

In [2]:
# MemoryStore extracts plan definitions from PricingConfigData
# via set_active_pricing().
store.set_active_pricing(
    PricingConfigData(
        models={
            "gpt-4o": "input_tokens * 5 + output_tokens * 15",
        },
        plans={
            "pro": PlanDefinition(
                id="pro", name="Pro Tier",
                free_allowance=50_000,
            ),
            "free": PlanDefinition(
                id="free", name="Free Tier",
                free_allowance=5_000,
            ),
        },
    ),
    label="default",
)
print("  Pricing config stored with 2 plan definitions")

  Pricing config stored with 2 plan definitions


### Assign a user and check allowance

In [3]:
user = str(uuid.uuid4())
store.set_user_plan(user, "pro")

allow = store.check_allowance(user)
print(f"  Plan:      {allow.plan_id}")
print(f"  Period:    {allow.period_start} → {allow.period_end}")
print(f"  Remaining: {allow.allowance_remaining}")
assert allow.allowance_remaining == 50_000
print("  ✓ Full 50 000 free allowance available")

  Plan:      pro
  Period:    2026-06-01T00:00:00 → 2026-06-30T00:00:00
  Remaining: 50000
  ✓ Full 50 000 free allowance available


### Consume allowance

In [4]:
store.increment_usage_window(user, "pro", 3_000)
allow2 = store.check_allowance(user)
print(f"  Remaining after 3 000 used: {allow2.allowance_remaining}")
assert allow2.allowance_remaining == 47_000
print("  ✓ Allowance correctly reduced")

  Remaining after 3 000 used: 47000
  ✓ Allowance correctly reduced


### Free tier vs pro tier

In [5]:
free_user = str(uuid.uuid4())
store.set_user_plan(free_user, "free")
free_allow = store.check_allowance(free_user)
print(f"  Free user allowance: {free_allow.allowance_remaining}")
assert free_allow.allowance_remaining == 5_000
print("  ✓ Free tier gets 5 000/month")

  Free user allowance: 5000
  ✓ Free tier gets 5 000/month
